# Flight Data Analysis and Preprocessing

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

file_path = "opensky.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ritikrao93/dva-cap",
  file_path,
)

df.head()

/tmp/ipykernel_7771/2499453664.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'dva-cap' dataset.


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,heading,vertical_rate,sensors,geo_altitude,squawk,spi,position_source,collection_time
0,511142,DMS1,Estonia,1.776336e+09,1776336263,17.6276,49.4815,2529.84,False,93.23,42.32,-3.90,NaN,2628.90,4370.0,False,0,2026-04-16 10:44:42.038813
1,39de4e,TVF49MD,France,1.776336e+09,1776336263,9.1655,42.7458,11277.60,False,229.36,126.15,-0.33,NaN,11513.82,2364.0,False,0,2026-04-16 10:44:42.038813
2,3ffc29,DMFSS,Germany,1.776336e+09,1776336263,11.7358,50.7950,1173.48,False,54.70,163.61,-0.33,NaN,NaN,NaN,False,0,2026-04-16 10:44:42.038813
3,ab1644,NaN,United States,1.776336e+09,1776336225,-74.1829,40.6824,NaN,True,1.29,90.00,NaN,NaN,NaN,NaN,False,0,2026-04-16 10:44:42.038813
4,80162d,AXB1527,India,1.776336e+09,1776336263,77.2991,25.8990,12192.00,False,234.17,1.64,0.00,NaN,12740.64,NaN,False,0,2026-04-16 10:44:42.038813


This section loads the flight data from a Kaggle dataset using `kagglehub`. The dataset, `ritikrao93/dva-cap`, contains information about aircraft movements. The `opensky.csv` file is loaded into a pandas DataFrame, and the first few rows are displayed to provide an initial overview of the data structure.

---

In [ ]:
df.shape

(108022, 18)

## 2. Initial Data Inspection

This section performs an initial inspection of the loaded dataset to understand its basic characteristics, such as dimensions, data types, and the presence of null values. This is a crucial step before any data cleaning or analysis.

### 2.1 Column Dtypes & Memory

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108022 entries, 0 to 108021
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   icao24           108022 non-null  object 
 1   callsign         105889 non-null  object 
 2   origin_country   108022 non-null  object 
 3   time_position    106857 non-null  float64
 4   last_contact     108022 non-null  int64  
 5   longitude        106857 non-null  float64
 6   latitude         106857 non-null  float64
 7   baro_altitude    97477 non-null   float64
 8   on_ground        108022 non-null  bool   
 9   velocity         108016 non-null  float64
 10  heading          108022 non-null  float64
 11  vertical_rate    97750 non-null   float64
 12  sensors          0 non-null       float64
 13  geo_altitude     96308 non-null   float64
 14  squawk           68520 non-null   float64
 15  spi              108022 non-null  bool   
 16  position_source  108022 non-null  int6

The `df.info()` method provides a concise summary of the DataFrame, including the number of entries, column names, non-null counts, and data types for each column. This helps in identifying columns with missing values and assessing memory usage.

### 2.2 Null Audit

In [ ]:
df.isnull().sum()

,0
icao24,0
callsign,2133
origin_country,0
time_position,1165
last_contact,0
longitude,1165
latitude,1165
baro_altitude,10545
on_ground,0
velocity,6


This step quantifies the number of null values in each column. It's essential to understand the extent of missing data to plan appropriate imputation or handling strategies.

### 3.4 Duplicate Check

Duplicate rows in a polling-based feed typically indicate repeated collection across overlapping polling windows. We quantify — but do not yet drop — to inform the cleaning sequence.


In [ ]:
df.duplicated().sum()

np.int64(0)

This cell checks for duplicate rows in the dataset. While duplicated rows might indicate repeated data collection, for now, we only quantify their presence to inform later cleaning steps.

### 2.3 Column Glossary

---

### 3.1 Drop Entirely-Null Columns

## 3. Data Cleaning and Preprocessing

This section focuses on cleaning and preprocessing the raw flight data to prepare it for analysis. This involves handling missing values, standardizing string formats, correcting data types, and ensuring physical consistency of the data points.

In [ ]:
# Drop columns where every row is NaN — zero information content
df = df.dropna(axis=1, how='all')
print(f"Remaining columns: {df.columns.tolist()}")

Remaining columns: ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'heading', 'vertical_rate', 'geo_altitude', 'squawk', 'spi', 'position_source', 'collection_time']


Columns that are entirely empty (i.e., every row is `NaN`) provide no useful information. This step automatically identifies and drops such columns to reduce data size and complexity.

### 3.2 String Normalisation

In [ ]:
str_cols = ['icao24', 'callsign', 'origin_country']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].str.strip()

This step ensures that string-based columns like `icao24`, `callsign`, and `origin_country` are cleaned by removing leading/trailing whitespace, which can cause inconsistencies and issues during analysis or aggregation.

### 3.3 Type Casting — `collection_time`

In [ ]:
df['collection_time'] = pd.to_datetime(df['collection_time'], errors='coerce')
print(f"collection_time dtype: {df['collection_time'].dtype}")
print(f"NaT count: {df['collection_time'].isna().sum()}")

collection_time dtype: datetime64[ns]
NaT count: 0


The `collection_time` column is converted to a datetime object. The `errors='coerce'` argument handles any unparseable date formats by converting them to `NaT` (Not a Time), although in this case, no `NaT` values are found, indicating successful conversion for all entries.

### 3.4 Boolean Casting — `on_ground` & `spi`

In [ ]:
df["on_ground"] = df["on_ground"].astype('boolean')
df["spi"] = df["spi"].astype("boolean")

The `on_ground` and `spi` columns, which represent boolean flags, are explicitly cast to pandas' `boolean` dtype. This ensures memory efficiency and proper handling of boolean logic.

### 3.5 Normalise Empty Callsigns to `NaN`

In [ ]:
df["callsign"] = df["callsign"].replace("", np.nan)

Empty strings in the `callsign` column are replaced with `np.nan` to standardize missing values. This allows for consistent handling of missing callsigns later in the cleaning process.

### 3.6 Drop Rows with Missing Coordinates

In [ ]:
rows_before = len(df)
df = df.dropna(subset=["latitude", "longitude"])
print(f"Dropped {rows_before - len(df)} rows with missing coordinates. Remaining: {len(df)}")

Dropped 1165 rows with missing coordinates. Remaining: 106857


Rows with missing `latitude` or `longitude` values are dropped. These coordinates are fundamental for tracking aircraft positions, and their absence makes the record largely unusable for geographical analysis.

### 3.7 Median Imputation — `velocity`, `baro_altitude`, `geo_altitude`

In [ ]:
df["velocity"] = df["velocity"].fillna(df["velocity"].median())
df["baro_altitude"] = df["baro_altitude"].fillna(df["baro_altitude"].median())
df["geo_altitude"] = df["geo_altitude"].fillna(df["geo_altitude"].median())

print("Post-imputation null counts:")
print(df[["velocity", "baro_altitude", "geo_altitude"]].isnull().sum())

Post-imputation null counts:
velocity         0
baro_altitude    0
geo_altitude     0
dtype: int64


Missing values in `velocity`, `baro_altitude`, and `geo_altitude` are imputed using the median of their respective columns. Median imputation is chosen to be robust against outliers, which are common in flight data.

### 3.8 Fill Missing `origin_country` with "Unknown"

In [ ]:
df["origin_country"] = df["origin_country"].fillna("Unknown")

Any remaining missing values in the `origin_country` column are filled with the string "Unknown." This ensures that all entries in this categorical column have a value, preventing issues in subsequent categorical analysis.

### 3.9 Drop Low-Signal Columns — `sensors`, `spi`, `squawk`

In [ ]:
df = df.drop(columns=['sensors', 'spi', 'squawk'], errors='ignore')
print(f"Remaining columns: {df.columns.tolist()}")
df.isnull().sum()

Remaining columns: ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'heading', 'vertical_rate', 'geo_altitude', 'position_source', 'collection_time']


,0
icao24,0
callsign,1741
origin_country,0
time_position,0
last_contact,0
longitude,0
latitude,0
baro_altitude,0
on_ground,0
velocity,0


Columns identified as having low signal or high sparsity (`sensors`, `spi`, `squawk`) are dropped. The `errors='ignore'` argument prevents errors if a column is already dropped or doesn't exist.

### 3.10 Drop `position_source`

In [ ]:
df = df.drop(columns=['position_source'], errors='ignore')
print(f"Remaining columns: {df.columns.tolist()}")

Remaining columns: ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'heading', 'vertical_rate', 'geo_altitude', 'collection_time']


The `position_source` column is also dropped, likely because it doesn't provide significant value for the intended analysis or has insufficient data quality.

### 3.11 `vertical_rate` — Domain-Constrained Fill + Row Pruning

In [ ]:
# Grounded aircraft have a physically defined vertical_rate of 0
df.loc[df['on_ground'] == True, 'vertical_rate'] = 0

# Airborne records with no vertical_rate cannot be safely imputed — drop them
rows_before = len(df)
df.dropna(subset=['vertical_rate'], inplace=True)
print(f"Dropped {rows_before - len(df)} airborne rows with null vertical_rate. Remaining: {len(df)}")

Dropped 62 airborne rows with null vertical_rate. Remaining: 106795


This step first sets the `vertical_rate` to 0 for all aircraft identified as `on_ground`, as this is a physically consistent value. Then, any remaining rows where `vertical_rate` is still null (implying airborne aircraft with unknown vertical movement) are dropped.

### 3.12 Callsign Fallback — Fill Remaining Nulls with "Unknown"

In [ ]:
df['callsign'] = df['callsign'].fillna('Unknown')

Finally, any remaining null values in the `callsign` column (after previous cleaning steps) are filled with "Unknown" to ensure data completeness for this field.

---

### 4.1 Null Audit — Final State

## 4. Post-Cleaning Verification

After all cleaning and preprocessing steps, this section performs a final audit to confirm the dataset's state. It checks for remaining null values, verifies the schema and shape, and performs consistency checks on physical parameters.

In [ ]:
null_report = df.isnull().sum()
print("=== Final Null Report ===")
print(null_report[null_report > 0] if null_report.sum() > 0 else "✓ No null values remain.")

=== Final Null Report ===
✓ No null values remain.


This audit confirms that no null values remain in the DataFrame after all the cleaning steps. A clean dataset is crucial for reliable analysis.

### 4.2 Schema & Shape Confirmation

In [ ]:
print(f"Final shape : {df.shape}")
print(f"Columns     : {df.columns.tolist()}")
print()
df.dtypes

Final shape : (106795, 14)
Columns     : ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'heading', 'vertical_rate', 'geo_altitude', 'collection_time']



,0
icao24,object
callsign,object
origin_country,object
time_position,float64
last_contact,int64
longitude,float64
latitude,float64
baro_altitude,float64
on_ground,boolean
velocity,float64


This provides a final overview of the DataFrame's shape (number of rows and columns) and the data types of all remaining columns, ensuring they are appropriate for further analysis.

### 4.3 Physical Consistency Checks

In [ ]:
checks = {
    "latitude"     : df["latitude"].between(-90, 90).all(),
    "longitude"    : df["longitude"].between(-180, 180).all(),
    "velocity"     : df["velocity"].between(0, 350).all(),
    "baro_altitude": df["baro_altitude"].between(-500, 15000).all(),
    "heading"      : df["heading"].between(0, 360).all(),
}

print("=== Physical Consistency Checks ===")
for feat, passed in checks.items():
    status = "✓ PASS" if passed else "✗ FAIL — investigate outliers"
    print(f"  {feat:<20} {status}")

=== Physical Consistency Checks ===
  latitude             ✓ PASS
  longitude            ✓ PASS
  velocity             ✗ FAIL — investigate outliers
  baro_altitude        ✗ FAIL — investigate outliers
  heading              ✓ PASS


Physical consistency checks validate that certain numeric features (`latitude`, `longitude`, `velocity`, `baro_altitude`, `heading`) fall within realistic and expected ranges. Any failures here indicate potential outliers or data errors that might need further investigation.

### 4.4 `on_ground` × `vertical_rate` Consistency

In [ ]:
grounded = df[df["on_ground"] == True]
inconsistent = grounded[grounded["vertical_rate"] != 0]
print(f"Grounded records with non-zero vertical_rate: {len(inconsistent)}")
if len(inconsistent) == 0:
    print("✓ All grounded aircraft have vertical_rate = 0.")

Grounded records with non-zero vertical_rate: 0
✓ All grounded aircraft have vertical_rate = 0.


This check specifically verifies that all aircraft marked as `on_ground` have a `vertical_rate` of 0, reinforcing the physical consistency of the data after imputation and cleaning.

---

### 5.1 Flight Phase Distribution

## 5. Exploratory Data Analysis (EDA) - Initial Insights

This section begins the exploratory data analysis, providing initial insights into the dataset's characteristics and distributions, which can help in understanding flight patterns and data composition.

In [ ]:
phase_counts = df["on_ground"].value_counts()
phase_pct    = df["on_ground"].value_counts(normalize=True) * 100

print("=== Flight Phase ===")
print(f"  Airborne : {phase_counts.get(False, 0):>6}  ({phase_pct.get(False, 0):.1f}%)")
print(f"  On Ground: {phase_counts.get(True,  0):>6}  ({phase_pct.get(True,  0):.1f}%)")

=== Flight Phase ===
  Airborne :  97002  (90.8%)
  On Ground:   9793  (9.2%)


This analyzes the distribution of aircraft being `on_ground` versus `airborne`, providing a fundamental understanding of the dataset's composition in terms of flight phases.

### 5.2 Top 10 Origin Countries

In [ ]:
top_countries = (
    df["origin_country"]
    .value_counts()
    .head(10)
    .rename_axis("country")
    .reset_index(name="aircraft_count")
)
print(top_countries.to_string(index=False))

       country  aircraft_count
 United States           30338
United Kingdom            7959
       Ireland            4358
       Germany            3828
        Turkey            3690
         Spain            3541
         China            3088
        France            3049
     Australia            2988
         Malta            2713


This identifies and displays the top 10 origin countries based on the count of aircraft records. It helps in understanding the geographical representation of the flight data.

### 5.3 Summary Statistics — Core Kinematic Features

In [ ]:
kinematic_cols = ["velocity", "baro_altitude", "geo_altitude", "vertical_rate", "heading"]
df[kinematic_cols].describe().round(2)

,velocity,baro_altitude,geo_altitude,vertical_rate,heading
count,106795.00,106795.00,106795.00,106795.00,106795.00
mean,162.86,7239.80,7504.96,0.08,186.84
std,84.74,4153.85,4197.08,4.90,102.13
min,0.00,-304.80,-53.34,-66.00,0.00
25%,84.36,3009.90,3238.50,-0.33,100.30
50%,201.24,8877.30,9235.44,0.00,196.28
75%,228.31,10965.18,11094.72,0.00,274.84
max,1259.64,17404.08,31211.52,106.64,359.87


Summary statistics (count, mean, std, min, 25%, 50%, 75%, max) are calculated for key kinematic features (`velocity`, `baro_altitude`, `geo_altitude`, `vertical_rate`, `heading`). This provides a quick quantitative overview of these critical flight parameters.

### 5.4 Altitude: Grounded vs Airborne

In [ ]:
alt_summary = df.groupby("on_ground")["baro_altitude"].describe().round(1)
print(alt_summary)

             count    mean     std    min     25%     50%      75%      max
on_ground                                                                  
False      97002.0  7106.1  4305.4 -106.7  2491.7  8938.3  10972.8  17404.1
True        9793.0  8564.1  1620.8 -304.8  8877.3  8877.3   8877.3  13045.4


This cell provides summary statistics for `baro_altitude` specifically, broken down by whether the aircraft is `on_ground` or `airborne`. This helps in understanding how altitude differs between these two flight phases.

---

In [ ]:
output_path = "opensky_cleaned.csv.gz"
df.to_csv(output_path, index=False, compression="gzip")
print(f"✓ Exported {len(df):,} rows × {df.shape[1]} columns → '{output_path}'")

✓ Exported 106,795 rows × 14 columns → 'opensky_cleaned.csv.gz'


## 6. Data Export

This final section exports the cleaned and preprocessed DataFrame to a compressed CSV file, making it ready for further analysis, model training, or sharing.

In [ ]:
df.isnull().sum()

,0
icao24,0
callsign,0
origin_country,0
time_position,0
last_contact,0
longitude,0
latitude,0
baro_altitude,0
on_ground,0
velocity,0


A final check for null values is performed to ensure that the exported DataFrame is completely free of missing data.

In [ ]:
df.shape

(106795, 14)

The final shape of the DataFrame is printed to confirm the number of rows and columns in the exported dataset.

---